In [6]:
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
import pandas as pd
import math
import time
# Tải dữ liệu
from ucimlrepo import fetch_ucirepo
phishing_websites = fetch_ucirepo(id=327)
X = phishing_websites.data.features
y = phishing_websites.data.targets

# Chia tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y['result'], test_size=0.2, random_state=42, stratify=y)

In [7]:
class Hyperband:
    def __init__(self, estimator, param_distributions, max_iter=81, eta=3, random_state=None):
        """ArithmeticError
        Khởi tạo Hyperband
        estimator: Mô hình
        param_distributions: Phân phối tham số
        max_iter: Số lần lặp tối đa
        eta: Hệ số giảm
        random_state: Ngẫu nhiên
        """
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.max_iter = max_iter  # Số lần lặp tối đa
        self.random_state = random_state # Lấy ngẫu nhiên
        self.eta = eta  # Hệ số giảm
        self.s_max = int(np.log(self.max_iter) / np.log(self.eta)) # Init s_max = log(max_iter)/log(eta)        
        self.B = (self.s_max + 1) * self.max_iter # Init B = (s_max + 1) * max_iter
        self.cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state) # Init cv = StratifiedKFold
        self.results = [] # Lưu các lần chạy
        self.total_runs = 0 # Đếm tổng số lần chạy
        if self.random_state is not None:
            np.random.seed(self.random_state)
    
    def sample_params(self):
        """
        Lấy mẫu tham số từ các phân phối hoặc danh sách giá trị.
        Hỗ trợ:
            - scipy.stats distributions (randint, uniform,...)
            - list giá trị rời rạc
        """
        sampled_params = {}
        for param, dist in self.param_distributions.items():
            sampled_params[param] = dist.rvs()
        return sampled_params
    
    def try_params_and_return_score(self, params, X, y):
        """
        Chạy cross validation với bộ tham số và trả về điểm số
        """
        self.estimator.set_params(**params)
        score = cross_val_score(estimator=self.estimator, X=X, y=y, cv=self.cv, scoring='accuracy', n_jobs=-1).mean()
        return score
    
    def sucessive_halving(self, s, n, r, X, y):
        """
        Thực hiện sucessive halving
        s: số lần lặp
        n: Số bộ tham số
        r: Số n_estimators tối đa
        X: tập dữ liệu
        y: tập dữ liệu
        """

        T = [self.sample_params() for _ in range(n)]
        param_id = [str(s) + '_'+ str(pid) for pid in list(range(n))]
        remaining_params = T.copy()
        remaining_params_id = param_id.copy()
        for i in range(s + 1):
            n_i = math.floor(n * self.eta ** (-i))
            r_i = int(r * self.eta ** i)
            print(i,n_i, r_i)
            scores = []
            for t, pid in zip(remaining_params,remaining_params_id):
                params = t.copy() # Dùng copy để tránh làm thay đổi các tham số của t
                # Tạo n_estimators dựa trên cấu hình hyperband
                params['n_estimators'] =  min(r_i, t['n_estimators'])
                # Chạy cross validation và trả về điểm số
                score = self.try_params_and_return_score(params, X, y)
                # Lưu kết quả
                scores.append(score)
                result = { 'param_id' : pid, 'params': t, 'score': score, 's': s, 'i': i, 'n_i': n_i, 'r_i': r_i}
                self.results.append(result)
                # Đếm tổng số lần chạy
                self.total_runs += 1

            # Chọn top k bộ tham số
            k = math.floor(n_i / n)
            top_k_indices = np.argsort(scores)[-k:][::-1]
            remaining_params = [remaining_params[i] for i in top_k_indices]
            remaining_params_id = [remaining_params_id[i] for i in top_k_indices]
        
        # Huấn luyện mô hình với tập tham số tốt nhất
        final_params = remaining_params[0]
        self.estimator.set_params(**final_params)
        final_score = self.try_params_and_return_score(final_params, X, y)
        return final_params,final_score
    
    def fit(self, X, y):
        """Thực hiện tối ưu hóa siêu tham số với Hyperband"""
        best_score = -np.inf
        best_params = None
        params_count = 0
        for s in reversed(range(self.s_max +1)):
            # Tính toán số lượng bộ tham số     
            n = math.ceil(self.B / self.max_iter * self.eta ** s / (s + 1))
            params_count += n
            # Tính toán số lượng n_estimators
            r = self.max_iter * self.eta ** (-s)
            # Thực hiện sucessive halving
            params  , score = self.sucessive_halving(s, n, r, X, y)
            # Cập nhật tham số tốt nhất
            if score > best_score:
                best_score = score
                best_params = params
        self.best_params_ = best_params
        self.best_score_ = best_score
        self.params_count = params_count

In [8]:
from scipy.stats import randint, uniform

# Định nghĩa không gian tham số rộng hơn cho LightGBM
param_distributions = {
    'num_leaves': randint(20, 100),  # Số lá trong cây
    'max_depth': randint(3, 12),     # Độ sâu tối đa
    'learning_rate': uniform(0.01, 0.3),  # Tốc độ học
    'n_estimators': randint(50, 243+1),
    'min_child_samples': randint(10, 50),  # Số mẫu tối thiểu trong mỗi lá
    'subsample': uniform(0.6, 0.4),   # Tỷ lệ mẫu sử dụng cho mỗi cây
    'colsample_bytree': uniform(0.6, 0.4),  # Tỷ lệ features sử dụng cho mỗi cây
    'reg_alpha': uniform(0, 1),       # L1 regularization
    'reg_lambda': uniform(0, 1),      # L2 regularization
    'min_child_weight': uniform(0, 1)  # Trọng số tối thiểu cho mỗi lá
}

In [9]:
from lightgbm import LGBMClassifier

In [10]:
start_time = time.time()
hb = Hyperband(
    estimator=LGBMClassifier(random_state=42, n_jobs=1),
    param_distributions=param_distributions,
    max_iter=81,
    eta=3,
    random_state=42
)

hb.fit(X_train, y_train)
hb_time = time.time() - start_time
print("Best parameters:", hb.best_params_)
print("Best score:", hb.best_score_)

0 81 1
1 27 3
2 9 9
3 3 27
4 1 81
0 34 3
1 11 9
2 3 27
3 1 81
0 15 9
1 5 27
2 1 81
0 8 27
1 2 81
0 5 81
Best parameters: {'num_leaves': 57, 'max_depth': 11, 'learning_rate': 0.26803671915676136, 'n_estimators': 151, 'min_child_samples': 17, 'subsample': 0.8897141232505648, 'colsample_bytree': 0.8996646556405132, 'reg_alpha': 0.14592207403514545, 'reg_lambda': 0.10279240449266569, 'min_child_weight': 0.5709045018441643}
Best score: 0.9690182095362821


In [4]:
print("Số lượng bộ siêu tham số", hb.params_count)

NameError: name 'hb' is not defined

In [ ]:

# RandomizedSearchCV
start_time = time.time()
random_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, n_jobs=1),
    param_distributions=param_distributions,
    n_iter= int(hb.params_count/2.5),
    cv=5,
    n_jobs=1,
    random_state=42,
    scoring='accuracy'
)
random_search.fit(X_train, y_train)
random_time = time.time() - start_time


[LightGBM] [Info] Number of positive: 3941, number of negative: 3134
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003356 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 89
[LightGBM] [Info] Number of data points in the train set: 7075, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.557032 -> initscore=0.229124
[LightGBM] [Info] Start training from score 0.229124
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

In [9]:
random_search.best_params_

{'colsample_bytree': 0.9467474821243366,
 'learning_rate': 0.2615442291291955,
 'max_depth': 10,
 'min_child_samples': 18,
 'min_child_weight': 0.8218600592903562,
 'n_estimators': 171,
 'num_leaves': 85,
 'reg_alpha': 0.08134878064189976,
 'reg_lambda': 0.08483771408519192,
 'subsample': 0.9946558314004702}

In [10]:
hb.best_params_

{'num_leaves': 69,
 'max_depth': 9,
 'learning_rate': 0.2076771825730454,
 'n_estimators': 114,
 'min_child_samples': 12,
 'subsample': 0.6924299186352285,
 'colsample_bytree': 0.8687570974394914,
 'reg_alpha': 0.019710537754364155,
 'reg_lambda': 0.10410858198457384,
 'min_child_weight': 0.7999160853731894}

In [ ]:
rd_s = LGBMClassifier(**random_search.best_params_, random_state=42)
rd_s.fit(X_train, y_train)
y_pred = rd_s.predict(X_test)
print(accuracy_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 4926, number of negative: 3918
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019018 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 89
[LightGBM] [Info] Number of data points in the train set: 8844, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.556988 -> initscore=0.228946
[LightGBM] [Info] Start training from score 0.228946
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [12]:
hb_s = LGBMClassifier(**hb.best_params_, random_state=42)
hb_s.fit(X_train, y_train)
y_pred = hb_s.predict(X_test)
print(accuracy_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 4926, number of negative: 3918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001306 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 89
[LightGBM] [Info] Number of data points in the train set: 8844, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.556988 -> initscore=0.228946
[LightGBM] [Info] Start training from score 0.228946
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf